**Nikhil Sunder**  
ICSRI Research Fellow, University of Miami | Open-Source Developer  
Herbert Business School, University of Miami  
B.S.B.A. Quantitative Economics & Finance; Minor in Mathematics  

**Correspondence:** nss106@miami.edu  
**Research profiles:** [GitHub](https://github.com/nikhilxsunder) · [ORCID](https://orcid.org/0009-0007-3323-1760) · [LinkedIn](https://www.linkedin.com/in/nikhil-sunder/) · [Zenodo](https://zenodo.org/search?q=metadata.creators.person_or_org.name:%22Sunder,+Nikhil%22)  
**Software:** [PyPI](https://pypi.org/user/nikhil.sunder/) · [Anaconda](https://anaconda.org/nikhil.sunder/)

# 11. Reinforcement learning: robustness and comparison

Every trained policy of notebooks `08` to `10`, side by side: closed-loop variances in the
Rudebusch-Svensson (1999) model, effective Taylor coefficients, and simulated paths and losses in the
economies the policies were trained in.

**Inputs:** `artifacts/rl/ddpg/`, `artifacts/rl/sac/`, `artifacts/rl/qlearning/` and the artifacts of
notebooks `00` to `07`. This notebook trains nothing.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import autofed as af
from autofed import plots, rl
from autofed.style import COLORS

nb = af.NotebookSession(
    "11_rl_robustness_comparison",
    number=11,
    title="Reinforcement learning: robustness and comparison",
)

lab = rl.PolicyLab.load(
    nb.workspace.artifacts
)  # economies of notebooks 01-06, settings of notebook 07
cfg = lab.config
OBS_SPECS = ["x1", "x2"]
LOSS_COLUMNS = {
    "dev2 pi": r"$\Delta^2(\pi^*, \pi_t)$",
    "dev2 y": r"$\Delta^2(y^*, y_t)$",
    "dev2 di": r"$\Delta^2(\Delta i_t)$",
    "penalty": "Penalty",
}
PATH_NOTE = (
    "Simulated from the first two estimation-window quarters under freshly drawn shocks; every"
    " policy faces the same shocks within an economy. Not a historical counterfactual: the Actual"
    " line shares only its first two quarters with the simulations."
)


def slug(name: str) -> str:
    return name.lower().replace("-", "_").replace(" ", "_")


print(
    "Economies (passed the transmission check of notebook 07):",
    ", ".join(lab.env_names),
    "| frames per run:",
    cfg.total_timesteps,
    "| seed:",
    cfg.seed,
)

ALGORITHMS = {"DDPG": "ddpg", "SAC": "sac", "Q-learning": "qlearning"}
cells = [
    cell
    for folder in ALGORITHMS.values()
    for cell in rl.load_cells(nb.artifacts("rl", folder))
    if cell.env_name in lab.env_names  # the economies the current configuration trains in
]
# Presentation order: economy as in notebook 07, then algorithm, observation and variant.
ALGO_ORDER = list(ALGORITHMS)
cells.sort(key=lambda c: (lab.env_names.index(c.env_name), ALGO_ORDER.index(c.algo), c.obs_spec, c.variant))
print(
    "Loaded", {algo: sum(c.algo == algo for c in cells) for algo in ALGORITHMS}, "trained cells"
)

Economies (passed the transmission check of notebook 07): SVAR, NARX | frames per run: 12500 | seed: 2026


ArtifactError: /Users/nikhilsunder/Documents/Github/University of Miami/Autonomous_Fed/notebooks/artifacts/rl/ddpg/ddpg_nssm_x1_linear.pt is not a trained cell of format version 2; re-run the algorithm notebook that trains it

## DSGE robustness exercise (Rudebusch-Svensson 1999)

### Methodology

Hinterlang & Tänzer (2021, Section 3.4 and Table 7) evaluate their optimised reaction functions
on 11 DSGE models from the Macroeconomic Model Database (Wieland et al., 2012, 2016) and report
the unconditional variances of inflation, the output gap, and the change in the policy rate
*relative to* the Taylor (1993) rule under each model. The exercise is a robustness check: a
policy rule that is optimised for one estimated economy (the SVAR or the ANN) is not guaranteed
to perform well in other plausible structural representations, and the DSGE comparison establishes
that the RL-optimised rules remain reasonably robust across models.

A full replication requires solving forward-looking rational-expectations DSGE models with Sims's
(2002) `gensys` or an equivalent (the Macroeconomic Model Database is built on `dynare`). Our
notebook does not embed `dynare`, so a strict replication of all 11 H&T models is out of scope.
We instead implement the *backward-looking* subset of the H&T robustness exercise, which reduces to a
closed-form solve. The canonical backward-looking benchmark in the monetary-policy literature is
Rudebusch & Svensson (1999, henceforth RS99), which H&T cite explicitly. We compute the unconditional
variance of each variable under the closed-loop dynamics induced by each candidate policy rule and
report the variances relative to the Taylor (1993) baseline.

This notebook evaluates every trained policy of notebooks `08` to `10`.

### Rudebusch-Svensson (1999) closed-form solver

The RS99 model, in deviations from steady state (Rudebusch & Svensson, 1999, eq. 1-2), is

$$\begin{aligned}
\pi_{t+1} &= 0.70\, \pi_t - 0.10\, \pi_{t-1} + 0.28\, \pi_{t-2} + 0.12\, \pi_{t-3} + 0.14\, y_t + \varepsilon^\pi_{t+1}, \\
y_{t+1} &= 1.16\, y_t - 0.25\, y_{t-1} - 0.10\, (\bar{\imath}_t - \bar{\pi}_t) + \varepsilon^y_{t+1}, \\
\bar{\imath}_t &= \tfrac{1}{4}(i_t + i_{t-1} + i_{t-2} + i_{t-3}), \qquad
\bar{\pi}_t = \tfrac{1}{4}(\pi_t + \pi_{t-1} + \pi_{t-2} + \pi_{t-3}),
\end{aligned}$$

with residual standard deviations $\sigma_\pi = 1.009$ and $\sigma_y = 0.819$. Both the nominal rate
and inflation enter the IS equation as four-quarter averages, and the inflation lags sum to one, so
the Phillips curve has a unit root: a rule with a long-run inflation response below one violates the
Taylor principle and leaves the closed loop unstable. We adopt the H&T policy rule format with one
lag, in deviations from steady state,

$$i_t = \beta_\pi^0 \pi_t + \beta_\pi^1 \pi_{t-1} + \beta_y^0 y_t + \beta_y^1 y_{t-1},$$

stack the system in the state
$z_t = (y_t, y_{t-1}, \pi_t, \pi_{t-1}, \pi_{t-2}, \pi_{t-3}, i_{t-1}, i_{t-2}, i_{t-3})$, whose three
rate lags carry the four-quarter average, substitute the policy rule, and solve for the closed-loop
$F(K)$ such that $z_{t+1} = F(K) z_t + \eta_{t+1}$. The unconditional state-covariance matrix is the
unique stable solution of the discrete-time Lyapunov equation $V = FVF^\top + \Sigma$. With nine
states the vectorised form $\mathrm{vec}(V) = (I - F \otimes F)^{-1}\mathrm{vec}(\Sigma)$ is an
81-equation linear system, solved directly with `torch.linalg.solve`
(`autofed.rl.build_rs99_F`, `rs99_variances`).

### Policy coefficients

Each trained policy enters through its `linearise_policy` coefficients, the OLS approximation of the
clipped policy on a grid over its observation. The slopes are the same in level and deviation form, so
they enter the log-linearised rule unchanged; $x^{(2)}$ policies contribute their fitted lag
coefficients $\beta_\pi^1$ and $\beta_y^1$. Values below 1 mean the rule lowers that variance relative
to the Taylor (1993) rule; `inf` marks an unstable closed loop, and $\rho$ is the spectral radius of
the closed-loop transition matrix (stable when $\rho < 1$).

In [ ]:
coefficients = {
    name: {"beta_pi_0": policy.rule.beta_pi, "beta_y_0": policy.rule.beta_y}
    for name, policy in lab.reference_rules().items()
}
groups = {name: "Reference rules" for name in coefficients}
for cell in cells:
    linear = rl.linearise_policy(cell)
    name = f"{cell.algo} | {cell.env_name}: {cell.obs_spec} {cell.variant}"
    coefficients[name] = {
        "beta_pi_0": linear["beta_pi"],
        "beta_pi_1": linear["beta_pi_1"],
        "beta_y_0": linear["beta_y"],
        "beta_y_1": linear["beta_y_1"],
    }
    groups[name] = cell.algo

rs99 = rl.rs99_table(coefficients)
rs99.index = pd.MultiIndex.from_tuples(
    [(groups[name], name.split(" | ")[-1]) for name in rs99.index],
    names=["Algorithm", "Policy"],
)
nb.table(
    rs99,
    "rs99_variance_ratios",
    "Closed-loop variances in the Rudebusch-Svensson model, relative to TR93",
    fontsize=8.0,
    note=(
        "Unconditional variances of inflation, the output gap and the change in the policy rate"
        " under each rule, divided by their values under TR93; Mean averages the three. inf:"
        " unstable closed loop; rho: spectral radius of the closed loop. With RS99's unit-root"
        " Phillips curve a rule whose long-run inflation response is below one is unstable."
        " Counterpart of Hinterlang & Tänzer (2021, Table 7) for the one backward-looking model."
    ),
)

In [ ]:
stable = rs99[np.isfinite(rs99["Mean"])]
fig, ax = plt.subplots(figsize=(8.6, 6.0))
markers = {
    "Reference rules": ("s", COLORS["black"]),
    "DDPG": ("o", COLORS["blue"]),
    "SAC": ("^", COLORS["purple"]),
    "Q-learning": ("D", COLORS["orange"]),
}
for group, (marker, color) in markers.items():
    part = stable[stable.index.get_level_values("Algorithm") == group]
    if part.empty:
        continue
    ax.scatter(
        part["var(pi)"],
        part["var(y)"],
        marker=marker,
        color=color,
        s=70 if group == "Reference rules" else 45,
        alpha=1.0 if group == "Reference rules" else 0.75,
        label=f"{group} ({len(part)})",
    )
for (group, name), row in stable.iterrows():
    ax.annotate(
        name,
        (row["var(pi)"], row["var(y)"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8.5 if group == "Reference rules" else 7.0,
        color=COLORS["black"] if group == "Reference rules" else COLORS["grey"],
    )
ax.axhline(1.0, color=COLORS["grey"], lw=0.7, ls="--")
ax.axvline(1.0, color=COLORS["grey"], lw=0.7, ls="--")
ax.set_xlabel("Variance of inflation, relative to TR93")
ax.set_ylabel("Variance of the output gap, relative to TR93")
ax.legend(loc="best")
nb.figure(
    fig,
    "rs99_variance_scatter",
    "Stable rules in the Rudebusch-Svensson model",
    note=(
        f"{len(stable)} of {len(rs99)} rules give a stable closed loop. Rules to the lower left"
        " of TR93 reduce both variances."
    ),
)

## Effective Taylor coefficients, all algorithms

In [ ]:
nb.table(
    rl.coefficient_table(cells),
    "taylor_coefficients",
    "Effective Taylor coefficients of every trained policy",
    fontsize=8.0,
    note=(
        "OLS of the clipped policy rate on a grid over the observation, in the level form of"
        " Hinterlang & Tänzer (2021, Table 4); x2 policies carry fitted lag coefficients and a"
        " policy that observes the previous rate is evaluated with it held at 4 percent. Grouped"
        " by the economy the policy was trained in."
    ),
)

## Simulated paths and losses, all algorithms

Each policy is simulated in the economy it was trained in. The figures show the reference rules and, for
each algorithm, the policy with the lowest loss in that economy; the table lists every policy.

In [ ]:
paths_by_economy = {
    env_name: lab.simulate_policies(env_name, cells) for env_name in lab.env_names
}
loss_table = lab.loss_table(paths_by_economy)

for env_name, paths in paths_by_economy.items():
    shown = {
        name: path for name, path in paths.items() if name in ("Actual", *lab.reference_rules())
    }
    for algo in ALGORITHMS:
        labels = [c.label for c in cells if c.env_name == env_name and c.algo == algo]
        best = loss_table.loc[env_name].loc[labels, "Loss"].idxmin()
        shown[best] = paths[best]
    fig = plots.policy_paths(shown, pi_star=cfg.pi_star, relative_time=True)
    nb.figure(
        fig,
        f"best_policies_{slug(env_name)}",
        f"Best policy of each algorithm and the reference rules in the {env_name} economy",
        note=PATH_NOTE
        + " For each algorithm the policy with the lowest loss in this economy is shown.",
    )

In [ ]:
nb.table(
    loss_table.rename(columns=LOSS_COLUMNS),
    "loss_decomposition",
    "Loss decomposition of every policy in its economy",
    fontsize=8.0,
    note=(
        "Mean squared deviations over the simulated quarters; Loss weights them as the reward"
        " does. Penalty is the mean footnote-13 term, so Loss + Penalty is minus the mean reward."
        " The Actual row is the same statistic on the realised data and is a reference level, not"
        " a like-for-like comparison."
    ),
)

### Discussion and limitations

**Faithful to H&T:**
- The economies come from the parent notebooks' fitted models directly, and only economies in which a
  sustained rate increase lowers inflation are used as policy environments (the transmission check of
  notebook `07`).
- The agent's action $i_t$ is mapped to $i_{t-1}$ in the *next* period's transition equation.
- Linear and nonlinear actor variants are plain `torch.nn` modules with H&T eq. 12-13 semantics: the
  ZLB $\max\{\cdot, 0\}$ sits inside the actor and there is no output squash. The one departure is a
  slope of $0.01$ below the bound instead of H&T's zero, which keeps a gradient alive; the realised
  rate is still exactly zero there.
- Both observation specifications $x_t^{(1)}$ and $x_t^{(2)}$ are trained for every applicable cell.
- Hidden-node search over critic and (nonlinear) actor nodes; agent selection by the H&T filter
  ($ER/ES > -4$, the band reached before $T$) followed by the steady-state-reward tie-break, the
  reward at the deterministic closed-loop fixed point.

**Pragmatic deviations from H&T:**
- Default architecture grid is narrower than H&T's full $\{1, \ldots, 10\}$ ranges. Setting
  `cfg.with_full_grid()` in notebook `07_rl_environments` recovers the full grid at an order-of-magnitude longer
  wall-clock.
- The TVP-SVAR-SV environment uses the smoothed posterior mean at the training endpoint
  (2007Q2). Because the parent notebook fits the TVP model on the full sample including out-of-sample
  data, this anchor incorporates a small amount of post-2007 information. Coefficients, the
  contemporaneous coefficient and the volatilities are posterior-mean plug-ins; the posterior draws
  are not propagated into the environment. Whether the anchor is used at all is decided by the
  transmission check.
- The reward function adds a Sack-Wieland $(\Delta i)^2$ smoothing term (cf. H&T eq. 16 which
  omits it); the previous rate is appended to the observation so that the term is a function of the
  observed state. Setting `omega_di = 0` recovers H&T's reward and observations exactly.
- A macro-state clamp at $\pm 10$ percentage points around steady state prevents runaway
  trajectories from dominating the agent-selection bookkeeping.
- The counterfactual paths are simulations under freshly drawn shocks with a common seed, not the
  historical counterfactual of H&T Section 3.3, which feeds the estimated structural shocks back
  through the economy. The "Actual" loss row is therefore a reference level, not a like-for-like
  comparison.

**DSGE exercise limitations:**
- We implement only the Rudebusch & Svensson (1999) backward-looking benchmark. The 10
  forward-looking models in H&T's exercise require solving rational-expectations equilibria
  via Sims's (2002) `gensys` or an equivalent, which we do not embed here. A full replication
  is a follow-on contribution.
- The nonlinear policies (DDPG nonlinear, SAC, Q-learning) are evaluated in the DSGE exercise
  using an OLS-linearised approximation of their reaction functions. The DSGE comparison requires
  linear feedback, and H&T exclude their nonlinear ANN policies from it for that reason, so these rows
  discard the precise nonlinearities of the learned policies.
- Most learned rules that are unstable in RS99 are so because their long-run inflation response is
  below one; RS99's unit-root Phillips curve makes the Taylor principle a necessary condition for
  stability, which the estimated economies the agents trained in do not impose.

# References

- Adam, K., & Billi, R. M. (2006). Optimal monetary policy under commitment with a zero bound on nominal interest rates. *Journal of Money, Credit and Banking*, 38(7), 1877-1905.
- Hinterlang, N., & Tänzer, A. (2021). Optimal monetary policy using reinforcement learning. *Deutsche Bundesbank Discussion Paper No. 51/2021*.
- Nikolsko-Rzhevskyy, A., Papell, D. H., & Prodan, R. (2018). The Taylor principles. *Journal of Macroeconomics*, 55, 14-30.
- Rudebusch, G. D., & Svensson, L. E. O. (1999). Policy rules for inflation targeting. In J. B. Taylor (Ed.), *Monetary Policy Rules* (pp. 203-262). University of Chicago Press.
- Sack, B., & Wieland, V. (2000). Interest-rate smoothing and optimal monetary policy: A review of recent empirical evidence. *Journal of Economics and Business*, 52(1-2), 205-228.
- Sims, C. A. (2002). Solving linear rational expectations models. *Computational Economics*, 20(1-2), 1-20.
- Taylor, J. B. (1993). Discretion versus policy rules in practice. *Carnegie-Rochester Conference Series on Public Policy*, 39, 195-214.
- Wieland, V., Afanasyeva, E., Kuete, M., & Yoo, J. (2016). New methods for macro-financial model comparison and policy analysis. In J. B. Taylor & H. Uhlig (Eds.), *Handbook of Macroeconomics* (Vol. 2, pp. 1241-1319). Elsevier.
- Wieland, V., Cwik, T., Müller, G. J., Schmidt, S., & Wolters, M. (2012). A new comparative approach to macroeconomic modeling and policy analysis. *Journal of Economic Behavior & Organization*, 83(3), 523-541.


# Export

The last cell converts this notebook to PDF in the shared `documents/` directory. A kernel can only
convert the copy saved on disk, so the cell waits briefly for the editor to save; if the file is still
behind, it writes nothing and asks you to save and run it again.

In [ ]:
nb.export_pdf()

/var/folders/cj/p5jdwnb923752l7zjc_69j5w0000gn/T/ipykernel_83930/281924391.py:1: UserWarning: 11_rl_robustness_comparison.ipynb on disk does not yet contain this run's outputs, so no PDF was written. Save the notebook and run this cell again.
  nb.export_pdf()
